In [2]:
from pyspark.sql import SparkSession

In [57]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_35/755975041.py:8: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [4]:
spark = SparkSession.builder \
    .appName("create mob tables") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "8")\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 13:07:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.sql("CREATE DATABASE IF NOT EXISTS gr_dm_sec")
spark.sql("USE gr_dm_sec")

DataFrame[]

In [10]:
df_base = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
)

In [15]:
cols = [
    # Основные ID
    F.col("msisdn"),
    F.col("msisdn").alias("app_n"),

    # Даты
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("tp_change_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("max_subs_fee_dt_day_31d"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("ep_tp_change_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("device_first_imei_msisdn_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("last_change_smartphone_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("start_use_cur_smartp_mod_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("mymts_max_activity_master_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("mymts_max_activity_slave_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("mnp_portation_in_mts_dt"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("last_active_roam_dt_day_90d"),

    # Числовые и категориальные признаки
    (F.rand() * 100000000 % 30).cast("int").alias("regid"),
    (F.rand() * 100000000 % 30).cast("int").alias("bill_group_name"),
    (F.rand() * 100000000 % 2000).cast("int").alias("lifetime"),
    (F.rand() * 100000000 % 50).cast("int").alias("m_categ_name"),
    (F.rand() * 100000000 % 20).cast("int").alias("m_reg_cd"),
    (F.rand() * 100000000 % 20).cast("int").alias("m_reg_name"),
    (F.rand() * 100000000 % 40).cast("int").alias("sale_channel_cd"),
    (F.rand() * 100000000 % 40).cast("int").alias("sale_channel_name"),
    (F.rand() * 100000000 % 2).cast("int").alias("fl_client_name_confirmed"),
    (F.rand() * 100000000 % 4).cast("int").alias("client_type_cd"),
    (F.rand() * 100000000 % 2).cast("int").alias("sex"),
    (F.rand() * 100000000 % 50 + 18).cast("int").alias("age"),
    (F.rand() * 100000000 % 10).cast("int").alias("citizenship_type"),
    (F.rand() * 100000000 % 10).cast("int").alias("foreign_citizenship_cd"),
    (F.rand() * 100000000 % 100).cast("int").alias("region_name"),
    (F.rand() * 100000000 % 25).cast("int").alias("msisdn_type_cd"),
    (F.rand() * 100000000 % 25).cast("int").alias("msisdn_cluster"),
    (F.rand() * 100000000 % 45).cast("int").alias("trust_category"),
    (F.rand() * 100000000 % 10001).cast("int").alias("credit_limit_value"),
    (F.rand() * 100000000 % 10).cast("int").alias("cnt_app_on_acc"),
    (F.rand() * 100000000 % 20).cast("int").alias("service_provider_cd"),
    (F.rand() * 100000000 % 2).cast("int").alias("fl_push_off"),
    (F.rand() * 100000000 % 2).cast("int").alias("fl_unlim_internet"),
    (F.rand() * 1000).cast("decimal(10,2)").alias("max_subs_fee_day_31d"),
    (F.rand() * 100000000 % 5).cast("int").alias("host_type"),
    (F.rand() * 1000).cast("int").alias("activity_on_bts_dur_day_30d"),
    (F.rand() * 10).cast("int").alias("cnt_give_out_prizes_day_7d"),
    (F.rand() * 20).cast("int").alias("cnt_give_out_prizes_day_30d"),
    (F.rand() * 50).cast("int").alias("cnt_give_out_prizes_day_90d"),
    (F.rand() * 100).cast("int").alias("cnt_give_out_prizes_day_180d"),

    # Строковые признаки
    F.concat(F.lit("tp_"), (F.rand() * 100).cast("int").cast("string")).alias("tp_name"),
    F.concat(F.lit("type_"), (F.rand() * 10).cast("int").cast("string")).alias("tp_type_name"),
    F.concat(F.lit("group_"), (F.rand() * 20).cast("int").cast("string")).alias("tp_group_name"),

    # Доп. числовые
    (F.rand() * 31).cast("int").alias("max_subs_fee_day_31d_days_ago"),
    (F.rand() * 1000).cast("int").alias("tp_current_use_dur"),
    (F.rand() * 2).cast("int").alias("tp_fl_unlim_internet"),
    (F.rand() * 5).cast("int").alias("tp_on_act_cd"),
    (F.rand() * 100).cast("int").alias("tp_prev_cd"),
    (F.rand() * 10000).cast("int").alias("ep_tp_id"),
    F.concat(F.lit("ep_"), (F.rand() * 100).cast("int").cast("string")).alias("ep_tp_name"),
    (F.rand() * 1000).cast("int").alias("ep_tp_change_days"),
    (F.rand() * 100).cast("decimal(10,2)").alias("ep_tp_cnt_gb"),
    (F.rand() * 5000).cast("int").alias("ep_tp_cnt_min"),
    (F.rand() * 1000).cast("int").alias("ep_tp_cnt_sms"),
    F.concat(F.lit("pay_"), (F.rand() * 5).cast("int").cast("string")).alias("ep_tp_pay_method_name"),
    F.concat(F.lit("prev_"), (F.rand() * 50).cast("int").cast("string")).alias("ep_tp_prev_name"),
    F.concat(F.lit("ch_"), (F.rand() * 10).cast("int").cast("string")).alias("ep_tp_service_channel_name"),
    (F.rand() * 3).cast("int").alias("ep_tp_status"),
    (F.rand() * 10).cast("int").alias("ep_tp_version"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppd_current"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppd_day_30d"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppd_day_90d"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppm_current"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppm_day_30d"),
    (F.rand() * 2).cast("int").alias("ep_tp_fl_ppm_day_90d"),

    # Устройства
    (F.rand() * 5 + 2).cast("int").alias("device_best_support_g"),
    (F.rand() * 1000).cast("int").alias("device_first_imei_msisdn_days"),
    F.concat(F.lit("model_"), (F.rand() * 100).cast("int").cast("string")).alias("device_model"),
    F.when(F.rand() < 0.5, F.lit("Android")).otherwise(F.lit("iOS")).alias("device_os_name"),
    F.when(F.rand() < 0.5, F.lit("Android")).otherwise(F.lit("iOS")).alias("device_os_name_prev"),
    F.concat(F.lit("v"), (F.rand() * 10 + 1).cast("int").cast("string")).alias("device_os_version"),
    (F.rand() * 5).cast("int").alias("device_type"),
    (F.rand() * 5).cast("int").alias("device_type_prev"),
    F.concat(F.lit("vendor_"), (F.rand() * 20).cast("int").cast("string")).alias("device_vendor"),
    (F.rand() * 10).cast("int").alias("cnt_smartphone_all_time"),
    (F.rand() * 1000).cast("int").alias("smartphone_use_dur_avg"),
    (F.rand() * 1000).cast("int").alias("last_change_smartphone_days"),
    (F.rand() * 1000).cast("int").alias("start_use_cur_smartp_mod_days"),

    # MyMTS
    (F.rand() * 2).cast("int").alias("mymts_fl_app_day_30d"),
    (F.rand() * 2).cast("int").alias("mymts_fl_activity_day_90d"),
    (F.rand() * 2).cast("int").alias("mymts_fl_wps"),
    (F.rand() * 2).cast("int").alias("mymts_fl_app_push_on"),
    (F.rand() * 2).cast("int").alias("mymts_fl_activity_master_day_30d"),
    (F.rand() * 2).cast("int").alias("mymts_fl_activity_master_day_90d"),
    (F.rand() * 1000).cast("int").alias("mymts_max_activity_master_days"),
    (F.rand() * 1000).cast("int").alias("mymts_max_activity_slave_days"),
    F.concat(F.lit("app_v"), (F.rand() * 10 + 1).cast("int").cast("string")).alias("mymts_version_app_str"),
    (F.rand() * 10 + 1).cast("int").alias("mymts_version_app"),

    # MNP
    (F.rand() * 2).cast("int").alias("fl_mnp_out"),
    (F.rand() * 2).cast("int").alias("fl_mnp_request"),
    (F.rand() * 1000).cast("int").alias("mnp_portation_in_mts_days"),

    # CC flags
    (F.rand() * 2).cast("int").alias("cc_fl_bill_group"),
    (F.rand() * 2).cast("int").alias("cc_fl_bill_group_b2b"),
    (F.rand() * 2).cast("int").alias("cc_fl_full"),
    (F.rand() * 2).cast("int").alias("cc_fl_full_b2b"),
    (F.rand() * 2).cast("int").alias("cc_fl_full_rtm"),
    (F.rand() * 2).cast("int").alias("cc_fl_m_categ"),
    (F.rand() * 2).cast("int").alias("cc_fl_m_categ_b2b"),
    (F.rand() * 2).cast("int").alias("cc_fl_other"),
    (F.rand() * 2).cast("int").alias("cc_fl_other_all"),
    (F.rand() * 2).cast("int").alias("cc_fl_other_b2b"),
    (F.rand() * 2).cast("int").alias("cc_fl_tp"),
    (F.rand() * 2).cast("int").alias("cc_fl_tp_b2b"),

    # Сигнал
    (F.rand() * 100).cast("decimal(5,2)").alias("prcnt_dur_2g_day_30d"),
    (F.rand() * 100).cast("decimal(5,2)").alias("prcnt_dur_3g_day_30d"),
    (F.rand() * 100).cast("decimal(5,2)").alias("prcnt_dur_4g_day_30d"),
    (F.rand() * 100).cast("decimal(5,2)").alias("prcnt_dur_no_signal_day_30d"),

    # SMS
    (F.rand() * 50).cast("int").alias("cnt_sms_vamzvonili_day_30d"),
    (F.rand() * 50).cast("int").alias("cnt_sms_yanasvyazi_day_30d"),

    # Активность
    (F.rand() * 2).cast("int").alias("fl_user_active_day_7d"),
    (F.rand() * 2).cast("int").alias("fl_user_active_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_signal_traffic_activity_day_1d"),
    (F.rand() * 2).cast("int").alias("fl_signal_traffic_activity_day_3d"),
    (F.rand() * 2).cast("int").alias("fl_signal_traffic_activity_day_7d"),
    (F.rand() * 2).cast("int").alias("fl_last_active_roam_day_90d"),
    (F.rand() * 90).cast("int").alias("last_active_roam_day_90d_days_ago"),

    (F.rand() * 2).cast("int").alias("fl_bound_card"),
    (F.rand() * 10).cast("int").alias("pay_method_cd"),
    F.concat(F.lit("pm_"), (F.rand() * 10).cast("int").cast("string")).alias("pay_method_name"),
    (F.rand() * 5).cast("int").alias("payer_type_cd"),
    F.concat(F.lit("pt_"), (F.rand() * 5).cast("int").cast("string")).alias("payer_type_name"),
    (F.rand() * 5).cast("int").alias("payment_card_type_day_30d"),
    (F.rand() * 5).cast("int").alias("payment_card_type_day_90d"),
    (F.rand() * 2).cast("int").alias("fl_ewallet_payment_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_ewallet_payment_day_90d"),
    (F.rand() * 2).cast("int").alias("fl_card_payment_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_card_payment_day_90d"),
    (F.rand() * 2).cast("int").alias("fl_cash_payment_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_cash_payment_day_90d"),
    (F.rand() * 31 + 1).cast("int").alias("expected_payment_payday"),
    (F.rand() * 2).cast("int").alias("fl_sbp_payment_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_sbp_payment_day_90d"),
    (F.rand() * 10).cast("int").alias("cnt_premium_paid_day_90d"),
    (F.rand() * 20).cast("int").alias("cnt_premium_paid_day_180d"),
    (F.rand() * 50).cast("int").alias("cnt_premium_paid_day_360d"),
    (F.rand() * 2).cast("int").alias("fl_unpaid_last_prompay"),
    (F.rand() * 10000).cast("decimal(10,2)").alias("sum_unpaid_last_prompay"),
    (F.rand() * 5).cast("int").alias("cnt_prompay_day_7d"),
    (F.rand() * 10).cast("int").alias("cnt_attempts_activation_prompay_day_7d"),
    (F.rand() * 5000).cast("decimal(10,2)").alias("sum_prompay_day_7d"),
    (F.rand() * 500).cast("decimal(10,2)").alias("clc_comission_prompay_day_7d"),
    (F.rand() * 500).cast("decimal(10,2)").alias("sum_paid_comission_prompay_day_7d"),
    (F.rand() * 500).cast("decimal(10,2)").alias("sum_unpaid_comission_prompay_day_7d"),
    (F.rand() * 20).cast("int").alias("cnt_prompay_day_30d"),
    (F.rand() * 30).cast("int").alias("cnt_attempts_activation_prompay_day_30d"),
    (F.rand() * 15).cast("int").alias("cnt_paid_prompay_day_30d"),
    (F.rand() * 15000).cast("decimal(10,2)").alias("sum_prompay_day_30d"),
    (F.rand() * 40).cast("int").alias("cnt_paid_prompay_day_90d"),
    (F.rand() * 40000).cast("decimal(10,2)").alias("sum_prompay_day_90d"),
    (F.rand() * 10).cast("int").alias("current_block_cd"),
    (F.rand() * 100).cast("int").alias("current_block_dur"),
    F.concat(F.lit("block_"), (F.rand() * 10).cast("int").cast("string")).alias("current_block_name"),
    (F.rand() * 10000).cast("decimal(10,2)").alias("unblock_limit"),
    (F.rand() * 2).cast("int").alias("fl_ban_calling"),
    (F.rand() * 2).cast("int").alias("fl_ban_mailing_email"),
    (F.rand() * 2).cast("int").alias("fl_ban_sms"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_cashback"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_kion"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_live"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_mts"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_music"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_payment"),
    (F.rand() * 2).cast("int").alias("fl_unsub_email_premium"),
    (F.rand() * 2).cast("int").alias("fl_email_comm_day_30d"),
    (F.rand() * 2).cast("int").alias("fl_email_comm_day_90d"),
    (F.rand() * 2).cast("int").alias("fl_email_comm_day_180d"),
    (F.rand() * 2).cast("int").alias("fl_email_comm_day_360d"),
    (F.rand() * 10).cast("int").alias("cnt_request_claim_day_30d"),
    (F.rand() * 20).cast("int").alias("cnt_request_day_30d"),
    (F.rand() * 1000).cast("int").alias("geo_day_bs_city"),
    (F.rand() * 10).cast("int").alias("geo_day_bs_city_popul_type"),
    (F.rand() * 1000).cast("int").alias("geo_night_bs_city"),
    (F.rand() * 10).cast("int").alias("geo_night_bs_city_popul_type"),
    (F.rand() * 100).cast("int").alias("geo_night_bs_district_msk"),
    (F.rand() * 100).cast("int").alias("geo_night_bs_region"),
    (F.rand() * 24 - 12).cast("int").alias("time_zone"),
    (F.rand() * 24 - 12).cast("int").alias("time_zone_geo"),
    F.date_add(F.lit("2020-01-01"), (F.rand() * 2000).cast("int")).alias("personal_data_block_date"),
    (F.rand() * 5).cast("int").alias("personal_data_status"),
    F.concat(F.lit("src_"), (F.rand() * 10).cast("int").cast("string")).alias("personal_data_verify_source_name"),
    (F.rand() * 10).cast("decimal(5,2)").alias("rate_comission"),
]

In [16]:
traffic_cols = [
    # Telegram
    (F.rand() * 1000).cast("decimal(10,2)").alias("tg_traf_vol_day_7d"),
    (F.rand() * 3000).cast("decimal(10,2)").alias("tg_traf_vol_day_30d"),
    (F.rand() * 8000).cast("decimal(10,2)").alias("tg_traf_vol_day_90d"),
    (F.rand() * 20000).cast("decimal(10,2)").alias("tg_traf_vol_day_360d"),

    # WhatsApp
    (F.rand() * 1000).cast("decimal(10,2)").alias("wa_traf_vol_day_7d"),
    (F.rand() * 3000).cast("decimal(10,2)").alias("wa_traf_vol_day_30d"),
    (F.rand() * 8000).cast("decimal(10,2)").alias("wa_traf_vol_day_90d"),
    (F.rand() * 20000).cast("decimal(10,2)").alias("wa_traf_vol_day_360d"),

    # WeChat
    (F.rand() * 800).cast("decimal(10,2)").alias("wc_traf_vol_day_7d"),
    (F.rand() * 2500).cast("decimal(10,2)").alias("wc_traf_vol_day_30d"),
    (F.rand() * 7000).cast("decimal(10,2)").alias("wc_traf_vol_day_90d"),
    (F.rand() * 18000).cast("decimal(10,2)").alias("wc_traf_vol_day_360d"),

    # Viber
    (F.rand() * 900).cast("decimal(10,2)").alias("vi_traf_vol_day_7d"),
    (F.rand() * 2800).cast("decimal(10,2)").alias("vi_traf_vol_day_30d"),
    (F.rand() * 7500).cast("decimal(10,2)").alias("vi_traf_vol_day_90d"),
    (F.rand() * 19000).cast("decimal(10,2)").alias("vi_traf_vol_day_360d"),

    # Snapchat
    (F.rand() * 700).cast("decimal(10,2)").alias("sc_traf_vol_day_7d"),
    (F.rand() * 2200).cast("decimal(10,2)").alias("sc_traf_vol_day_30d"),
    (F.rand() * 6000).cast("decimal(10,2)").alias("sc_traf_vol_day_90d"),
    (F.rand() * 15000).cast("decimal(10,2)").alias("sc_traf_vol_day_360d"),

    # VK
    (F.rand() * 1200).cast("decimal(10,2)").alias("vk_traf_vol_day_7d"),
    (F.rand() * 3500).cast("decimal(10,2)").alias("vk_traf_vol_day_30d"),
    (F.rand() * 9000).cast("decimal(10,2)").alias("vk_traf_vol_day_90d"),
    (F.rand() * 22000).cast("decimal(10,2)").alias("vk_traf_vol_day_360d"),

    # OK
    (F.rand() * 600).cast("decimal(10,2)").alias("ok_traf_vol_day_7d"),
    (F.rand() * 1800).cast("decimal(10,2)").alias("ok_traf_vol_day_30d"),
    (F.rand() * 5000).cast("decimal(10,2)").alias("ok_traf_vol_day_90d"),
    (F.rand() * 12000).cast("decimal(10,2)").alias("ok_traf_vol_day_360d"),

    # Instagram
    (F.rand() * 1100).cast("decimal(10,2)").alias("inst_traf_vol_day_7d"),
    (F.rand() * 3200).cast("decimal(10,2)").alias("inst_traf_vol_day_30d"),
    (F.rand() * 8500).cast("decimal(10,2)").alias("inst_traf_vol_day_90d"),
    (F.rand() * 21000).cast("decimal(10,2)").alias("inst_traf_vol_day_360d"),

    # X (Twitter)
    (F.rand() * 500).cast("decimal(10,2)").alias("x_traf_vol_day_7d"),
    (F.rand() * 1500).cast("decimal(10,2)").alias("x_traf_vol_day_30d"),
    (F.rand() * 4000).cast("decimal(10,2)").alias("x_traf_vol_day_90d"),
    (F.rand() * 10000).cast("decimal(10,2)").alias("x_traf_vol_day_360d"),

    # Facebook
    (F.rand() * 900).cast("decimal(10,2)").alias("fb_traf_vol_day_7d"),
    (F.rand() * 2700).cast("decimal(10,2)").alias("fb_traf_vol_day_30d"),
    (F.rand() * 7200).cast("decimal(10,2)").alias("fb_traf_vol_day_90d"),
    (F.rand() * 18000).cast("decimal(10,2)").alias("fb_traf_vol_day_360d"),

    # TikTok
    (F.rand() * 1300).cast("decimal(10,2)").alias("tt_traf_vol_day_7d"),
    (F.rand() * 3800).cast("decimal(10,2)").alias("tt_traf_vol_day_30d"),
    (F.rand() * 10000).cast("decimal(10,2)").alias("tt_traf_vol_day_90d"),
    (F.rand() * 25000).cast("decimal(10,2)").alias("tt_traf_vol_day_360d"),

    # Личный кабинет
    (F.rand() * 300).cast("decimal(10,2)").alias("lk_traf_vol_day_7d"),
    (F.rand() * 900).cast("decimal(10,2)").alias("lk_traf_vol_day_30d"),
    (F.rand() * 2500).cast("decimal(10,2)").alias("lk_traf_vol_day_90d"),
    (F.rand() * 6000).cast("decimal(10,2)").alias("lk_traf_vol_day_360d"),

    # YouTube
    (F.rand() * 1500).cast("decimal(10,2)").alias("yt_traf_vol_day_7d"),
    (F.rand() * 4500).cast("decimal(10,2)").alias("yt_traf_vol_day_30d"),
    (F.rand() * 12000).cast("decimal(10,2)").alias("yt_traf_vol_day_90d"),
    (F.rand() * 30000).cast("decimal(10,2)").alias("yt_traf_vol_day_360d"),

    # Rutube
    (F.rand() * 400).cast("decimal(10,2)").alias("rt_traf_vol_day_7d"),
    (F.rand() * 1200).cast("decimal(10,2)").alias("rt_traf_vol_day_30d"),
    (F.rand() * 3200).cast("decimal(10,2)").alias("rt_traf_vol_day_90d"),
    (F.rand() * 8000).cast("decimal(10,2)").alias("rt_traf_vol_day_360d"),

    # VK Video
    (F.rand() * 800).cast("decimal(10,2)").alias("vkvid_traf_vol_day_7d"),
    (F.rand() * 2400).cast("decimal(10,2)").alias("vkvid_traf_vol_day_30d"),
    (F.rand() * 6400).cast("decimal(10,2)").alias("vkvid_traf_vol_day_90d"),
    (F.rand() * 16000).cast("decimal(10,2)").alias("vkvid_traf_vol_day_360d"),

    # Twitter (дубль)
    (F.rand() * 500).cast("decimal(10,2)").alias("tw_traf_vol_day_7d"),
    (F.rand() * 1500).cast("decimal(10,2)").alias("tw_traf_vol_day_30d"),
    (F.rand() * 4000).cast("decimal(10,2)").alias("tw_traf_vol_day_90d"),
    (F.rand() * 10000).cast("decimal(10,2)").alias("tw_traf_vol_day_360d"),

    # Kion
    (F.rand() * 600).cast("decimal(10,2)").alias("kion_traf_vol_day_7d"),
    (F.rand() * 1800).cast("decimal(10,2)").alias("kion_traf_vol_day_30d"),
    (F.rand() * 4800).cast("decimal(10,2)").alias("kion_traf_vol_day_90d"),
    (F.rand() * 12000).cast("decimal(10,2)").alias("kion_traf_vol_day_360d"),

    # KP.RU
    (F.rand() * 200).cast("decimal(10,2)").alias("kp_traf_vol_day_7d"),
    (F.rand() * 600).cast("decimal(10,2)").alias("kp_traf_vol_day_30d"),
    (F.rand() * 1600).cast("decimal(10,2)").alias("kp_traf_vol_day_90d"),
    (F.rand() * 4000).cast("decimal(10,2)").alias("kp_traf_vol_day_360d"),

    # СТС
    (F.rand() * 300).cast("decimal(10,2)").alias("st_traf_vol_day_7d"),
    (F.rand() * 900).cast("decimal(10,2)").alias("st_traf_vol_day_30d"),
    (F.rand() * 2400).cast("decimal(10,2)").alias("st_traf_vol_day_90d"),
    (F.rand() * 6000).cast("decimal(10,2)").alias("st_traf_vol_day_360d"),

    # Wi-Fi
    (F.rand() * 2000).cast("decimal(10,2)").alias("wi_traf_vol_day_7d"),
    (F.rand() * 6000).cast("decimal(10,2)").alias("wi_traf_vol_day_30d"),
    (F.rand() * 16000).cast("decimal(10,2)").alias("wi_traf_vol_day_90d"),
    (F.rand() * 40000).cast("decimal(10,2)").alias("wi_traf_vol_day_360d"),

    # Okko
    (F.rand() * 700).cast("decimal(10,2)").alias("okko_traf_vol_day_7d"),
    (F.rand() * 2100).cast("decimal(10,2)").alias("okko_traf_vol_day_30d"),
    (F.rand() * 5600).cast("decimal(10,2)").alias("okko_traf_vol_day_90d"),
    (F.rand() * 14000).cast("decimal(10,2)").alias("okko_traf_vol_day_360d"),

    # IVI
    (F.rand() * 650).cast("decimal(10,2)").alias("ivi_traf_vol_day_7d"),
    (F.rand() * 1950).cast("decimal(10,2)").alias("ivi_traf_vol_day_30d"),
    (F.rand() * 5200).cast("decimal(10,2)").alias("ivi_traf_vol_day_90d"),
    (F.rand() * 13000).cast("decimal(10,2)").alias("ivi_traf_vol_day_360d"),

    # Premier
    (F.rand() * 550).cast("decimal(10,2)").alias("pr_traf_vol_day_7d"),
    (F.rand() * 1650).cast("decimal(10,2)").alias("pr_traf_vol_day_30d"),
    (F.rand() * 4400).cast("decimal(10,2)").alias("pr_traf_vol_day_90d"),
    (F.rand() * 11000).cast("decimal(10,2)").alias("pr_traf_vol_day_360d"),

    # Amazon
    (F.rand() * 400).cast("decimal(10,2)").alias("am_traf_vol_day_7d"),
    (F.rand() * 1200).cast("decimal(10,2)").alias("am_traf_vol_day_30d"),
    (F.rand() * 3200).cast("decimal(10,2)").alias("am_traf_vol_day_90d"),
    (F.rand() * 8000).cast("decimal(10,2)").alias("am_traf_vol_day_360d"),

    # Netflix
    (F.rand() * 900).cast("decimal(10,2)").alias("nf_traf_vol_day_7d"),
    (F.rand() * 2700).cast("decimal(10,2)").alias("nf_traf_vol_day_30d"),
    (F.rand() * 7200).cast("decimal(10,2)").alias("nf_traf_vol_day_90d"),
    (F.rand() * 18000).cast("decimal(10,2)").alias("nf_traf_vol_day_360d"),
]

In [17]:
all_cols = cols + traffic_cols

In [18]:
df_cp_mobile = df_base.select(*all_cols)

In [19]:
(
    df_cp_mobile
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable('gr_dm_sec.cp_mobile_daily_history')
)

26/03/07 13:39:05 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [20]:
spark.table('gr_dm_sec.cp_mobile_daily_history').count()

2479161

In [26]:
spark.sql("CREATE DATABASE IF NOT EXISTS ff_dm")
spark.sql("USE ff_dm")

DataFrame[]

In [27]:
spark.sql("CREATE DATABASE IF NOT EXISTS gr_dm")
spark.sql("USE gr_dm")

DataFrame[]

In [28]:
df_model_sex = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()

    .withColumn('model_sex', (F.rand() * 10000 % 2).cast("int"))
    .withColumn('business_dt', F.current_date())
)

In [29]:
(
    df_model_sex
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable('ff_dm.scoring_customer_sex')
)

In [30]:
df_model_age = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()

    .withColumn('model_age', (F.rand() * 10000 % 50 + 18).cast("int"))
    .withColumn('business_dt', F.current_date())
)

In [31]:
(
    df_model_age
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable('ff_dm.scoring_customer_age')
)

In [33]:
df_model_income = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()

    .withColumn('model_income', (F.rand() * 1000000 % 1000000).cast("int"))
    .withColumn('business_dt', F.current_date())
)

In [34]:
(
    df_model_income
    .repartition(1)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable('gr_dm.dm_model_income__hist')
)

In [35]:
spark.sql("CREATE DATABASE IF NOT EXISTS spfix_dic")
spark.sql("USE spfix_dic")

DataFrame[]

In [45]:
spark.sql("CREATE DATABASE IF NOT EXISTS spfix_dm")
spark.sql("USE spfix_dm")

DataFrame[]

In [52]:
spark.sql("CREATE DATABASE IF NOT EXISTS spfix_sb")
spark.sql("USE spfix_sb")

DataFrame[]

In [36]:
competitor_call_data = [
    ("rt", "4951234567"),
    ("domru", "4952345678"),
    ("beeline", "4953456789"),
    ("megafon", "4954567890"),
    ("rt", "4991112233"),
    ("domru", "4992223344"),
]

In [37]:
competitor_call_df = spark.createDataFrame(competitor_call_data, ["competitor", "phone_num"])
(
    competitor_call_df
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dic.competitor_call")
)

In [38]:
competitor_host_data = [
    ("rt", "rostelecom.ru"),
    ("domru", "domru.ru"),
    ("beeline", "beeline.ru"),
    ("megafon", "megafon.ru"),
    ("rt", "rt.ru"),
    ("domru", "dom.ru"),
]

In [39]:
competitor_host_df = spark.createDataFrame(competitor_host_data, ["competitor", "host name"])
(
    competitor_host_df
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dic.competitor_host")
)

In [44]:
calls = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
    .limit(534691)
    .join(spark.table("spfix_dic.competitor_call"))
    .withColumn('call_cnt', (F.rand() * 1000000 % 30).cast("int"))
)

In [43]:
calls

DataFrame[msisdn: int, competitor: string, phone_num: string]

In [46]:
(
    calls
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.join_to_cnum")
)

In [47]:
sms = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
    .limit(534691)
    .join(spark.table("spfix_dic.competitor_call"))
    .limit(534691)
    .withColumn('call_sms', (F.rand() * 1000000 % 5).cast("int"))
)

In [48]:
(
    sms
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.join_to_cnum_sms")
)

In [49]:
web = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
    .limit(1275129)
    .join(spark.table("spfix_dic.competitor_host"))
    .limit(2000000)
    .withColumn('request_cnt', (F.rand() * 1000000 % 100).cast("int"))
    .withColumn('request_days', (F.rand() * 1000000 % 30).cast("int"))
)

In [50]:
web.count()

2000000

In [51]:
(
    web
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("spfix_dm.join_to_clickstream")
)

In [53]:
spark.sql("CREATE DATABASE IF NOT EXISTS netscout_cdm")
spark.sql("USE netscout_cdm")

DataFrame[]

In [54]:
def mock_getZidCentroidUDF(col_name):
    from pyspark.sql import functions as F
    return F.struct(
        (F.rand() * 10 + 54).cast("double").alias("lat"),   # ~54–64° (Россия)
        (F.rand() * 20 + 30).cast("double").alias("lon")    # ~30–50°
    )

In [55]:
logical_date = "2026-02-28"
logical_date_obj = datetime.strptime(logical_date, '%Y-%m-%d').date()
part_dt_0 = datetime(logical_date_obj.year, logical_date_obj.month, 22).date()

In [58]:
dates = []
for month_ago in range(4):
    cur_part_dt = part_dt_0 - relativedelta(months=month_ago)
    dates.append(cur_part_dt)

In [59]:
msisdn_df = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
)

In [60]:
all_data = []

for dt in dates:
    df_day = (
        msisdn_df
        .withColumn("business_dt", F.lit(dt))
        # Имитация geo_area_key как случайного ID
        .withColumn("refined_home_place_grid_geo_area_key", (F.rand() * 1000000).cast("long"))
        .withColumn("refined_work_place_grid_geo_area_key", (F.rand() * 1000000).cast("long"))
    )
    all_data.append(df_day)

In [61]:
full_df = all_data[0]
for df in all_data[1:]:
    full_df = full_df.union(df)

In [62]:
full_df.count()

9916644

In [64]:
full_df = (
    full_df
    .withColumn('home_place_coord', 
    F.struct(
        (F.rand() * 10 + 54).cast("double").alias("lat"),
        (F.rand() * 20 + 30).cast("double").alias("lon")
    )
               )
                
    .withColumn('work_place_coord', 
    F.struct(
        (F.rand() * 10 + 54).cast("double").alias("lat"),
        (F.rand() * 20 + 30).cast("double").alias("lon")
    )
    )
)

In [65]:
full_df.count()

9916644

In [66]:
(
    full_df
    .repartition(2)
    .write
    .mode("overwrite")
    .format("orc")
    .saveAsTable("netscout_cdm.agg_subs_geo_home_work")
)

In [67]:
business_month_end = "2026-02-28"

In [68]:
categories = spark.createDataFrame([(i,) for i in range(1, 96)], ["category_id"])

In [75]:
df_full = (
    spark.table('mtsfix_dm.agg_mtsfix_red_convergent_fixa_a')
    .union(spark.table('mtsfix_dm.agg_mtsfix_red_convergent_mgts_fixa_a'))
    .select('foris_msisdn')
    .withColumnRenamed('foris_msisdn', 'msisdn')
    .distinct()
    .crossJoin(categories)
    .limit(20_356_091)

    .withColumn("regid", (F.rand() * 10000 % 100).cast("double"))
    .withColumn("app_n", (F.rand() * 100000000000 % 10_000_000).cast("double"))
    .withColumn("num_requests_to_overall_by_user", (F.rand() * 100).cast("double"))
    .withColumn("num_active_days_to_overall_by_user", (F.rand() * 30).cast("double"))
    .withColumn("num_requests_to_mean_by_category", (F.rand() * 50).cast("double"))
    .withColumn("num_active_days_to_mean_by_category", (F.rand() * 15).cast("double"))
    .withColumn("business_dt", F.lit(business_month_end))
)

In [76]:
(
    df_full
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .mode("overwrite")
    .format("orc")
    .saveAsTable("gr_dm.feature_store_hosts_user_ratio_stats")
)

In [74]:
df_full

DataFrame[msisdn: int, category_id: bigint, regid: double, app_n: double, num_requests_to_overall_by_user: double, num_active_days_to_overall_by_user: double, num_requests_to_mean_by_category: double, num_active_days_to_mean_by_category: double, business_dt: string]

In [72]:
df_full.count()

20000000

In [70]:
df_full.count()

235520295

In [23]:
!hdfs dfs -du -s -h hdfs://namenode:9000/user/hive/warehouse/gr_dm_sec.db/cp_mobile_daily_history

/usr/bin/sh: 1: hdfs: not found


In [24]:
location = spark.sql("DESCRIBE FORMATTED gr_dm_sec.cp_mobile_daily_history") \
    .filter(F.col("col_name") == "Location") \
    .select("data_type").collect()[0][0]

In [25]:
location

'hdfs://namenode:9000/user/hive/warehouse/gr_dm_sec.db/cp_mobile_daily_history'